# Profiling CYTools Steps for chi=-6 Polytopes
Each cell times one step. Run sequentially.

In [ ]:
# Cell 1: Fetch one polytope
import time, numpy as np, sys
print(sys.executable)
t0 = time.time()
from cytools import fetch_polytopes
from cytools.config import enable_experimental_features
enable_experimental_features()
polytopes = list(fetch_polytopes(chi=-6, lattice='N', limit=1))
p = polytopes[0]
print(f'FETCH: {time.time()-t0:.3f}s  n_verts={len(p.vertices())}')

In [ ]:
# Cell 2: Triangulate + CY
t1 = time.time()
tri = p.triangulate()
cy = tri.get_cy()
print(f'TRI+CY: {time.time()-t1:.3f}s  h11={cy.h11()} h21={cy.h21()}')

In [ ]:
# Cell 3: Divisor basis, intersection numbers, c2
t2 = time.time()
db = np.asarray(cy.divisor_basis())
print(f'  div_basis shape={db.shape}  ({time.time()-t2:.3f}s)')

t3 = time.time()
intnums = cy.intersection_numbers()
print(f'  intnums entries={len(intnums)}  ({time.time()-t3:.3f}s)')

t4 = time.time()
c2 = np.array(cy.second_chern_class(), dtype=float)
print(f'  c2 len={len(c2)}  ({time.time()-t4:.3f}s)')

In [ ]:
# Cell 4: Kahler cone + hyperplanes (CHEAP)
t5 = time.time()
kc = cy.toric_kahler_cone()
print(f'  kahler_cone: {time.time()-t5:.3f}s')

t6 = time.time()
hyps = np.array(kc.hyperplanes(), dtype=float)
print(f'  hyperplanes: {time.time()-t6:.3f}s  n_mori={len(hyps)}')
print(f'\n  >>> kc.rays() is the suspected bottleneck — test in next cell')

In [ ]:
# Cell 5: kc.rays() — THE SUSPECTED BOTTLENECK
# Time it to see how long it actually takes for h11=13
t7 = time.time()
rays = np.array(kc.rays(), dtype=float)
print(f'  kc.rays(): {time.time()-t7:.3f}s  n_rays={len(rays)}')

In [ ]:
# Cell 6: Chi computation helpers setup
if db.ndim == 2:
    n_toric = db.shape[1]
    def to_toric(bv): return np.asarray(bv, dtype=float) @ db.astype(float)
else:
    n_toric = max(int(db.max()) + 1, len(c2))
    def to_toric(bv):
        v = np.zeros(n_toric)
        for a, val in enumerate(bv):
            v[int(db[a])] = val
        return v

c2v = c2.copy()
if len(c2v) < n_toric:
    c2v = np.pad(c2v, (0, n_toric - len(c2v)))

def chi_lb(n_vec):
    n = np.array(n_vec, dtype=float)
    if len(n) < n_toric: n = np.pad(n, (0, n_toric - len(n)))
    elif len(n) > n_toric: n = n[:n_toric]
    cubic = 0.0
    for (a,b,c), kval in intnums.items():
        if max(a,b,c) >= n_toric: continue
        if a==b==c: cubic += kval * n[a]**3
        elif a==b: cubic += 3*kval*n[a]**2*n[c]
        elif b==c: cubic += 3*kval*n[a]*n[b]**2
        elif a==c: cubic += 3*kval*n[a]**2*n[b]
        else: cubic += 6*kval*n[a]*n[b]*n[c]
    return cubic/6 + np.dot(c2v[:n_toric], n[:n_toric])/12

print(f'n_toric={n_toric}, n_intnums={len(intnums)}')
# Single chi evaluation time
t_single = time.time()
for _ in range(100):
    chi_lb(np.ones(n_toric))
dt_chi = (time.time() - t_single) / 100
print(f'Single chi eval: {dt_chi*1000:.3f}ms')

In [ ]:
# Cell 7: Chi on Mori generators (cheap approach)
t10 = time.time()
mori_chis = []
for h in hyps:
    ht = to_toric(h)
    mori_chis.append(chi_lb(ht))
print(f'Chi on {len(hyps)} Mori gens: {time.time()-t10:.3f}s')
mori_chis = np.array(mori_chis)
print(f'  chi range: [{mori_chis.min():.2f}, {mori_chis.max():.2f}]')

In [ ]:
# Cell 8: brentq on Mori generators
from scipy.optimize import brentq

def find_t_star(ray_toric, target=3.0):
    def f(t): return chi_lb(t * ray_toric) - target
    test_ts = np.linspace(0.01, 5.0, 30)
    try:
        vals = [f(t) for t in test_ts]
        for j in range(len(vals)-1):
            if vals[j]*vals[j+1] < 0:
                return brentq(f, test_ts[j], test_ts[j+1], xtol=1e-10)
    except: pass
    return None

t11 = time.time()
n_mori_found = 0
for h in hyps:
    ht = to_toric(h)
    t = find_t_star(ht)
    if t is not None: n_mori_found += 1
print(f'brentq on {len(hyps)} Mori: {time.time()-t11:.3f}s  found={n_mori_found}')

In [ ]:
# Cell 9: Integer lattice search near Mori generators
t12 = time.time()
n_chi3 = 0
seen = set()
scales = [0.5, 0.75, 1.0, 1.25, 1.5, 2.0, 2.5, 3.0]
h11 = cy.h11()

for gen in hyps:
    for s in scales:
        base = tuple(np.round(s * gen).astype(int))
        candidates = [base]
        for d in range(h11):
            p1 = list(base); p1[d] += 1; candidates.append(tuple(p1))
            m1 = list(base); m1[d] -= 1; candidates.append(tuple(m1))
        for D_tuple in candidates:
            if D_tuple in seen: continue
            seen.add(D_tuple)
            D_basis = np.array(D_tuple, dtype=float)
            D_toric = to_toric(D_basis)
            chi_val = chi_lb(D_toric)
            if abs(chi_val - 3.0) < 1e-6:
                pairings = hyps @ D_basis
                min_mori = int(round(np.min(pairings)))
                n_chi3 += 1
                if n_chi3 <= 5:
                    print(f'  chi=3: D={D_basis[:6]}... min_mori={min_mori}')

print(f'\nInt search ({len(seen)} pts): {time.time()-t12:.3f}s  chi=3 found={n_chi3}')

In [ ]:
# Cell 10: Summary
print('='*60)
print('PROFILING SUMMARY')
print('='*60)
print(f'  h11 = {h11}')
print(f'  n_mori = {len(hyps)}')
print(f'  n_rays = {len(rays)}')
print(f'  n_intnums = {len(intnums)}')
print(f'')
print(f'  CONCLUSION: Skip kc.rays(), use Mori-only approach')
print(f'  This eliminates the vertex enumeration bottleneck')
print(f'  and reduces ray-based search from {len(rays)} to {len(hyps)} directions')